# 🧠 ระบบพิกัดกรอบขอบเขต (Bounding Box Coordinate Systems): รูปแบบ, คณิตศาสตร์ และการแปลงค่า

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Bounding Box Coordinate Systems**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายการแสดงพิกัดแบบมุม (XYXY) และแบบจุดกึ่งกลาง (XYWH)
2. พัฒนาฟังก์ชันการแปลงค่าพิกัดจากศูนย์ด้วย NumPy
3. เขียนสูตรคณิตศาสตร์ของการปรับสัดส่วนมาตรฐาน (scale-normalization) ที่ใช้ในไฟล์ข้อความอธิบายข้อมูล (annotation text files) ของ YOLO
4. รันกระบวนการแปลงพิกัดครบวงจร: พิกัดมุมดิบ -> พิกัดกึ่งกลางดิบ -> ปรับมาตรฐาน YOLO -> พิกัดมุมดิบ
5. แสดงผลภาพกรอบขอบเขตบนระนาบพิกัด พร้อมทำเครื่องหมายมุม, ความกว้าง, ความสูง และจุดกึ่งกลาง
6. เชื่อมโยงรูปแบบพิกัดแบบปรับมาตรฐานเข้ากับการขยายข้อมูลหลายระดับขนาด (multi-scale data augmentations) ในการฝึกโมเดล YOLO

เรามาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลยครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Set seed for reproducibility
np.random.seed(42)

## 1. การสร้างฟังก์ชันการแปลงพิกัดใน NumPy

เราจะพัฒนาการแปลงพิกัดแบบเวกเตอร์ (vectorized conversions) ที่สามารถจัดการได้ทั้งกรอบเดียวหรือหลายกรอบพร้อมกัน (อาร์เรย์รูปทรง $(N, 4)$) ครับ

In [ ]:
def xywh_to_xyxy(boxes):
    boxes = np.array(boxes)
    converted = np.zeros_like(boxes, dtype=float)
    converted[..., 0] = boxes[..., 0] - (boxes[..., 2] / 2.0)
    converted[..., 1] = boxes[..., 1] - (boxes[..., 3] / 2.0)
    converted[..., 2] = boxes[..., 0] + (boxes[..., 2] / 2.0)
    converted[..., 3] = boxes[..., 1] + (boxes[..., 3] / 2.0)
    return converted

def xyxy_to_xywh(boxes):
    boxes = np.array(boxes)
    converted = np.zeros_like(boxes, dtype=float)
    converted[..., 0] = (boxes[..., 0] + boxes[..., 2]) / 2.0
    converted[..., 1] = (boxes[..., 1] + boxes[..., 3]) / 2.0
    converted[..., 2] = boxes[..., 2] - boxes[..., 0]
    converted[..., 3] = boxes[..., 3] - boxes[..., 1]
    return converted

def normalize_xywh(boxes_xywh, width, height):
    boxes_xywh = np.array(boxes_xywh, dtype=float)
    normed = np.zeros_like(boxes_xywh)
    normed[..., 0] = boxes_xywh[..., 0] / width
    normed[..., 1] = boxes_xywh[..., 1] / height
    normed[..., 2] = boxes_xywh[..., 2] / width
    normed[..., 3] = boxes_xywh[..., 3] / height
    return normed

def denormalize_xywh(boxes_norm, width, height):
    boxes_norm = np.array(boxes_norm, dtype=float)
    denormed = np.zeros_like(boxes_norm)
    denormed[..., 0] = boxes_norm[..., 0] * width
    denormed[..., 1] = boxes_norm[..., 1] * height
    denormed[..., 2] = boxes_norm[..., 2] * width
    denormed[..., 3] = boxes_norm[..., 3] * height
    return denormed

## 2. การตรวจสอบการแปลงพิกัดแบบครบวงจร (Round-Trip Verification)

เราจะจำลองรูปภาพที่มีความละเอียด $640 \times 480$ พิกเซล และมีวัตถุอยู่ที่พิกัดมุม $[100, 150, 300, 400]$ ดังนี้:
-   แปลงพิกัดมุม (XYXY) ไปเป็นแบบจุดกึ่งกลาง (XYWH)
-   ปรับมาตรฐานพิกัดจุดกึ่งกลางให้อยู่ในรูปแบบของ YOLO
-   แปลงพิกัดกลับมาเป็นค่าพิกเซลดิบ (denormalize)
-   แปลงกลับไปเป็นพิกัดมุม (XYXY) อีกครั้งและตรวจสอบความถูกต้องเชิงตำแหน่ง

In [ ]:
W, H = 640, 480
box_xyxy = [100, 150, 300, 400]

box_xywh = xyxy_to_xywh(box_xyxy)
print("Centroid Format (XYWH)   :", box_xywh)

box_norm = normalize_xywh(box_xywh, W, H)
print("Normalized YOLO Format    :", box_norm)

box_denorm = denormalize_xywh(box_norm, W, H)

box_restored = xywh_to_xyxy(box_denorm)
print("Restored Corners (XYXY)   :", box_restored)
print("Reconstructed perfectly?  :", np.allclose(box_xyxy, box_restored))

## 3. การแสดงผลภาพพารามิเตอร์ของกรอบขอบเขต

เรามาวาดรูปกรอบขอบเขตภายในพื้นที่พิกัดขนาด $640 \times 480$ พร้อมทำเครื่องหมายจุดและแกนต่างๆ กันครับ

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.set_xlim(0, W)
ax.set_ylim(H, 0)

x1, y1, x2, y2 = box_xyxy
xc, yc, w, h = box_xywh
rect = patches.Rectangle((x1, y1), w, h, linewidth=3, edgecolor='green', facecolor='none')
ax.add_patch(rect)

ax.scatter(xc, yc, color='red', s=120, zorder=5, label=f'Centroid (xc={xc}, yc={yc})')

ax.text(x1 - 10, y1 - 15, f'Top-Left Corners\n(x1={x1}, y1={y1})', color='darkgreen', fontsize=10, ha='right')
ax.text(x2 + 10, y2 + 25, f'Bottom-Right Corners\n(x2={x2}, y2={y2})', color='darkgreen', fontsize=10, ha='left')
ax.text(xc + 15, yc - 15, f'w={w}, h={h}', color='blue', fontsize=11, fontweight='bold')

plt.title('Bounding Box Geometry (Image Pixel Space)')
plt.xlabel('X coordinate')
plt.ylabel('Y coordinate')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

สังเกตพล็อตที่ได้ครับ:
- จุดมุมบนซ้าย $(x_1, y_1)$ อยู่ที่ตำแหน่ง $(100, 150)$
- จุดมุมล่างขวา $(x_2, y_2)$ อยู่ที่ตำแหน่ง $(300, 400)$
- ความกว้างเท่ากับ $200$ พิกเซล ความสูงเท่ากับ $250$ พิกเซล และจุดกึ่งกลางอยู่ที่ตำแหน่ง $(200, 275)$
- แกน y มีค่าเริ่มต้นเป็น 0 ที่ด้านบนสุด ซึ่งสอดคล้องกับการกำหนดดัชนีของภาพดิจิทัล (digital image indexing)

## 💡 การเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **ความคงทนต่อการปรับขนาด (Scale Invariance):** ทำไม YOLO ถึงใช้ค่าปรับมาตรฐาน (มีค่าระหว่าง `0` ถึง `1`) ในไฟล์ข้อความป้ายกำกับแทนที่จะใช้พิกัดพิกเซลจริง? นั่นเป็นเพราะในระหว่างการฝึกฝน YOLO จะใช้การขยายข้อมูล (augmentations) หลากหลายรูปแบบ เช่น การปรับขนาดและการย่อขยายความละเอียดของภาพ (เช่น ฝึกที่ขนาด 640x640 แต่วัดผลที่ขนาด 320x320)
*   หากป้ายกำกับใช้ค่าพิกเซลพิกัดจริง เราจำเป็นต้องปรับขนาดพิกัดด้วยตนเองสำหรับทุกๆ การขยายข้อมูล แต่เนื่องจากค่าปรับมาตรฐานนั้นเป็นอิสระจากขนาดของเฟรมภาพ YOLO จึงสามารถคูณค่าทศนิยมของป้ายกำกับด้วยขนาดรูปภาพปัจจุบันได้โดยตรง ทำให้การปรับสัดส่วนเกิดขึ้นอย่างอัตโนมัติและมีความคงทนสูง!